# Ebook RAG con LlamaIndex

Inspirado en [huggingface](https://huggingface.co/learn/cookbook/en/rag_llamaindex_librarian)

## Dependencias

Para correr los modelos de manera local.

In [ ]:
%pip install -q \
    llama-index \
    EbookLib \
    html2text \
    llama-index-embeddings-huggingface \
    llama-index-llms-ollama \
    llama-index-readers-file

Instalar Ollama de https://ollama.com/download/OllamaSetup.exe

In [1]:
import subprocess, time, urllib.request

def ollama_is_up(timeout: float = 1.0) -> bool:
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=timeout)
        return True
    except Exception:
        return False

if not ollama_is_up():
    print("Starting Ollama server...")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        # hides the console window on Windows; harmless on other OSes
        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
    )
    for _ in range(60):
        if ollama_is_up():
            print("Ollama is up.")
            break
        time.sleep(1)
    else:
        raise RuntimeError("Ollama did not become ready in 60s.")
else:
    print("Ollama is already running.")

Ollama is already running.


Pull Llama2 from the Ollama library.

In [ ]:
!ollama pull llama2

## Documentos

Descargamos epubs de la pagina [Project Gutenberg](https://www.gutenberg.org/ebooks/subject/10163), bajo la tematica "Filosofia de la Religion" (que esten en ingles).

In [ ]:
from pathlib import Path
import urllib.request
import shutil

library_dir = Path("./test/library")
library_dir.mkdir(parents=True, exist_ok=True)

books = {
    "elementary_forms_religious_life-Durkheim.epub":         "https://www.gutenberg.org/ebooks/41360.epub.noimages",
    "essence_christianity-Feuerbach.epub":                   "https://www.gutenberg.org/ebooks/47025.epub.noimages",
    "religious_sentiment-Brinton.epub":                      "https://www.gutenberg.org/ebooks/30061.epub.noimages",
    "philosophy_natural_theology-Jackson.epub":              "https://www.gutenberg.org/ebooks/47314.epub.noimages",
    "myth_science-Vignoli.epub":                             "https://www.gutenberg.org/ebooks/17802.epub.noimages",
    "authorize_bible_interpeters-Beecher.epub":              "https://www.gutenberg.org/ebooks/56354.epub.noimages",
    "sources_religious_insight-Royce.epub":                  "https://www.gutenberg.org/ebooks/33677.epub.noimages",
    "philosophy_religion_psychology_history-Sabatier.epub":  "https://www.gutenberg.org/ebooks/38446.epub.noimages",
    "god_world-Robinson.epub":                               "https://www.gutenberg.org/ebooks/30709.epub.noimages",
    "common_sense_religion-Beecher.epub":                    "https://www.gutenberg.org/ebooks/55531.epub.noimages",
}

for name, url in books.items():
    dest = library_dir / name
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[skip] {name}")
        continue
    print(f"[dl]   {name}")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as resp, open(dest, "wb") as f:
        shutil.copyfileobj(resp, f)

print("Done.")

## LlamaIndex

RAG with LlamaIndex, at its core, consists of the following broad phases:

1. **Loading**, de los documentos;
2. **Indexing**, vector embeddings;
3. **Querying**, para recuper los documentos relevantes.


### Loading

Naturally, let's start with the **loading** phase.

El objeto 
[`SimpleDirectoryReader`](https://docs.llamaindex.ai/en/stable/module_guides/loading/simpledirectoryreader.html)
soporta archivos en epub.

In [27]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import EpubReader
import random

loader = SimpleDirectoryReader(
    input_dir="./test/",
    recursive=True,
    required_exts=[".epub"],
    file_extractor={".epub": EpubReader()},
)

documents = loader.load_data()

len(documents)

10

In [36]:
for doc in random.choices(documents,k=2):  # Check the first couple of documents
    print("="*20)
    txt = doc.text
    n = len(txt)
    L = 1000
    a = random.randint(101, n-L-1)
    print(txt[:100])
    print(txt[a:a+L])

## The Project Gutenberg eBook of Outlines of a Philosophy of Religion based
on Psychology and Histo
n the concept of _the good_ are
insignificant and worthless because neither man nor the good of man can be the
measure of things? If this remark is useful for abating human pride and
preventing childish illusions, it does not efface the primordial distinction
between good and evil inherent to the human mind, nor would one wish to deduce
from it the vanity of all morality, and the equal worth of all the
manifestations of life. The proof, moreover, that the rule of _the good_ is
above man is that it judges and condemns him pitilessly; it is that
consciousness, independently of the painful or agreeable sensations that it
receives from things, establishes between them a fitness (_convenance_), a
hierarchy, and constitutes the harmonious unity of the universe itself in the
supreme idea of the sovereign good. If the legitimacy of the confidence which
the conscience has in its rule is to be co

### Indexing

Se usa 
[`BAAI/bge-small-en-v1.5`](https://huggingface.co/BAAI/bge-base-en-v1.5) to
para generar los embeddings.


In [37]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embedding_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

2026-11-12 19:01:21,700 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5 "HTTP/1.1 200 OK"
2026-11-12 19:01:21,846 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1008.99it/s]
2026-11-12 19:01:22,779 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-11-12 19:01:22,877 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-11-12 19:01:23,288 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/2_Normalize/config.json "HTTP/1.1 404 Not Found"
2026-11-12 19:01:23,320 - INFO - Loaded 1 prompt with these keys: ['query']


In [38]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embedding_model,
)

In [39]:
from llama_index.core.vector_stores import SimpleVectorStore

vector_store = index.storage_context.vector_store
if isinstance(vector_store, SimpleVectorStore):
    for node_id in vector_store.data.embedding_dict:
        embedding = vector_store.get(node_id)
        print(f"Node ID: {node_id}, Dim: {len(embedding)}, vector: {embedding}")

Node ID: 7088e3b1-0f94-4331-9653-593143bb934a, Dim: 384, vector: [-0.07684482634067535, 0.02879560925066471, -0.05044684186577797, -0.007528099697083235, 0.04525936022400856, -0.025354741141200066, -0.018078060820698738, -0.027316821739077568, -0.03416621685028076, 0.025348378345370293, 0.009329119697213173, 0.01575406640768051, -0.00878813024610281, -0.06900707632303238, 0.01952151581645012, 0.005435910541564226, -0.012155558913946152, 0.004889166913926601, -0.05207612365484238, 0.02439369075000286, 0.01795547641813755, -0.009583796374499798, 0.025908004492521286, -0.03649130091071129, 0.0029642435256391764, 0.019223954528570175, -0.029424408450722694, 0.007979673333466053, -0.041527602821588516, -0.1402389258146286, -0.0337807834148407, -0.03824977949261665, 0.023276325315237045, 0.012642933987081051, 0.021805379539728165, 0.05540967360138893, -0.015472136437892914, 0.0017586869653314352, 0.007951712235808372, 0.006033731158822775, 0.025921549648046494, 0.05861203372478485, 0.0243947

In [41]:
retriever = index.as_retriever(similarity_top_k=5)
nodes = retriever.retrieve("community and economics")

for node_with_score in nodes:
    print(f"Score: {node_with_score.score:.4f}")
    print(node_with_score.node.get_content()[:200])
    print("---")

Score: 0.7158
Our institutions have the
hearty co-operation of the religious bodies where they are located.

"At our last annual meeting, an urgent request was made  to the Association to
aid in the establishment o
---
Score: 0.7080
From
this has come the struggle to multiply colleges and female seminaries in each
of the several denominations. Each is now acting _as a sect_ in starting new
institutions all over the land, that dem
---
Score: 0.7000
In the face of these two opposed objections the mind
remains uncertain. But if the social origin of the categories is admitted, a
new attitude becomes possible, which we believe will enable us to esca
---
Score: 0.6904
They are a source of insight to all who know of their life, and who rightly
appreciate its meaning. And of such is the kingdom of loyalty. And the
communities which such men form and serve are essenti
---
Score: 0.6834
In New York City another
lady has endowed a theological professorship. In Albany, New York, a lady has
given $

### Querying

Para esto se usara Llama 2.


In [42]:
from llama_index.llms.ollama import Ollama

llama = Ollama(
    model="llama2",
    request_timeout=1800.0, #30 minutos
)

query_engine = index.as_query_engine(llm=llama)

2026-11-12 19:46:23,318 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


## Prueba


In [43]:
print(query_engine.query("What are the titles of all the books available? Show me the context used to derive your answer."))

2026-11-12 19:53:06,888 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Based on the context information provided, the following are the titles of the books available in the library:

1. "Essence of Christianity-Feuerbach.epub"
2. "A History of Materialism" by Professor F. A. Lange
3. "Natural Law: An Essay in Ethics" by Edith Simcox
4. "The Creed of Christendom: Its Foundations Contrasted with Its Superstructures" by W. R. Greg
5. "Outlines of the History of Religion to the Spread of the Universal Religions" by C. P. Tiele

The context used to derive the answer is the list of books provided in the file path, which includes the titles of the books available in the library.


In [44]:
print(query_engine.query("What is religion?"))

2026-11-12 20:00:16,706 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Based on the context provided, religion can be defined as a complex and multifaceted concept that has evolved over time and across cultures. The query challenges the traditional notion of religion as a belief in supernatural powers and entities, and instead offers a more nuanced understanding of religion as a set of practices, beliefs, and experiences that are rooted in the human desire to make sense of the world and our place in it.

The context highlights the limitations of the traditional definition of religion as a belief in gods or spiritual beings, and instead argues for a more inclusive understanding of religion that encompasses a wide range of beliefs, practices, and experiences. This is evident in the discussion of Buddhism, which is presented as a religion that rejects the idea of a god or supreme being, yet still provides a moral framework and a sense of meaning and purpose to its adherents.

In conclusion, the query challenges the traditional definition of religion and offe